# Tunix-Med: Final Knowledge Evaluation

This notebook re-evaluates the **fine-tuned `google/gemma-3-1b-it` model** after training with **Tunix**. We compare its performance against the baseline to quantify the knowledge gain in the medical domain.

### Evaluation Pipeline
We use the same 25 cardiology questions as the baseline evaluation.

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = info_device()
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

BASE_MODEL = "google/gemma-3-1b-it"
ADAPTER_PATH = "tunix-medical-model" # Path where Tunix saved the adapter

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype, device_map=device)

# Load the Tunix adapter (Peft format)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model = model.merge_and_unload() # Merge for faster inference

## Final Evaluation Loop
We re-run the evaluation from notebook 01 and compare metrics.